In [2]:
import os
import torch
import clip
import pandas as pd
import numpy as np
from PIL import Image
from nsd_access import NSDAccess
from tqdm import tqdm

In [6]:
stim_descriptions = pd.read_csv('../data/nsddata/experiments/nsd/nsd_stim_info_merged.csv', index_col=0)
images = torch.load("../data/preprocessed_data/images_73k.pt").requires_grad_(False).to("cpu")

# First URL: This is the original read-only NSD file path (The actual data)
# Second URL: Local files that we are adding to the dataset and need to access as part of the data
# Object for the NSDAccess package
nsda = NSDAccess('/home/naxos2-raid25/kneel027/home/surly/kendrick-data/nsd', '/home/naxos2-raid25/kneel027/home/kneel027/nsd_local')
captions = nsda.read_image_coco_info([i for i in range(73000)], info_type='captions', show_annot=False)
# Load the CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-L/14", device=device)
captions_array = np.empty(73000, dtype=object)


loading annotations into memory...
Done (t=1.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!


In [18]:

for i in tqdm(range(0, 73000)):
    image = Image.fromarray(images[i].numpy().reshape((425, 425, 3)).astype(np.uint8))
    
    image_tensor = preprocess(image).unsqueeze(0).to(device)
    # Variable to store the captions of the image. 
    prompts = []
    
    # Load the 5 prompts for each image
    for j in range(len(captions[i])):
        # Index into the caption list and get the corresponding 5 captions. 
        prompts.append(captions[i][j]['caption'])
    
    # Encode the image
    # ...

    # Tokenize and preprocess the prompts
    text_tokens = clip.tokenize(prompts).to(device)
    # Compute image and text embeddings
    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(text_tokens)

    # Normalize the embeddings
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    
    # Compute the similarity scores
    similarity = (100.0 * image_features @ text_features.T).squeeze(0).cpu().numpy()
    best_caption_idx = similarity.argmax()
    best_caption = prompts[best_caption_idx]
    captions_array[i] = best_caption
    # tqdm.write(f"Best prompt: {best_caption}")
        

100%|██████████| 73000/73000 [29:00<00:00, 41.94it/s]


In [20]:
np.save("../data/preprocessed_data/captions_73k.npy", captions_array)

In [6]:
captions_array = np.load("../data/preprocessed_data/captions_73k.npy", allow_pickle=True)

In [8]:
for i in [28751, 30856, 53881, 61177, 65872]:
    print(captions_array[i])

A surfer raises his arms to keep his balance on the wave
a pair of boats in the water next to a beach
A brown and blue bird is standing on the dirt.
Two people with umbrellas at the top of a cliff.
Two nicely decorated donuts sit in a pink box on the table. 


In [9]:
file_path = '/home/naxos2-raid25/kneel027/home/kneel027/Second-Sight/data/nsddata_stimuli/stimuli/imagery_images/captions.txt'

# Read the text file and extract the strings
with open(file_path, 'r') as file:
    captions = [line.strip() for line in file.readlines()]
print(captions)
# Create a numpy array from the captions
captions_array = np.array(captions)

# Save the numpy array as captions.npy
np.save('/home/naxos2-raid25/kneel027/home/kneel027/Second-Sight/data/nsddata_stimuli/stimuli/imagery_images/captions.npy', captions_array)


['A horizontal black bar in front of a gray background', 'A diagonal black bar oriented towards the top right of a gray background', 'A vertical black bar in front of a gray background', 'A diagonal black bar oriented towards the top left of a gray background', 'A black plus symbol in front of a gray background', 'A black X symbol in front of a gray background', 'A surfer raises his arms to keep his balance on the wave', 'a pair of boats in the water next to a beach', 'A brown and blue bird is standing on the dirt.', 'Two people with umbrellas at the top of a cliff.', 'Two nicely decorated donuts sit in a pink box on the table.', 'Two women in white dresses stand in front of a floral wallpapered wall', 'stripes', 'zebra', 'mammal', 'yellow', 'banana', 'fruit']
